# Ejercicio 9: Uso de la API de Google Gemini

### Nombre: Anthony Goyes

En este ejercicio vamos a aprender a utilizar la API de Google Gemini

In [2]:
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os
import numpy as np
import pandas as pd

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("No se encontró GEMINI_API_KEY en el archivo .env")

client = genai.Client(api_key=api_key)

print("Cliente de Gemini creado correctamente")

Cliente de Gemini creado correctamente


## 1. Uso básico

El siguiente código sirve para conectarse con la API de Google Gemini de forma básica

In [3]:
prompt = "Explica brevemente qué es un Sistema de Recuperación de Información."

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
    config=types.GenerateContentConfig(
        temperature=0.2
    )
)

print("Respuesta de Gemini:")
print(response.text)

Respuesta de Gemini:
Un **Sistema de Recuperación de Información (SRI)** es un sistema diseñado para **encontrar y recuperar información relevante** de una gran colección de documentos o datos, en respuesta a una consulta o necesidad de información de un usuario.

Su objetivo principal es ayudar a los usuarios a acceder a la información que buscan, filtrando el ruido y presentando los resultados más pertinentes.

**El ejemplo más conocido es un motor de búsqueda web (como Google).** Otros ejemplos incluyen catálogos de bibliotecas, sistemas de búsqueda en bases de datos empresariales o la búsqueda de productos en tiendas online.


## 2. Retrieval

### 2.1 Cargo el corpus de 20 News Groups

Se utiliza el corpus 20 Newsgroups, que contiene documentos de diferentes categorías temáticas. Para este ejercicio se seleccionan algunas categorías y se trabaja con una muestra pequeña para evitar demasiadas llamadas a la API.

In [4]:
from sklearn.datasets import fetch_20newsgroups

categories = [
    "sci.space",
    "rec.sport.baseball",
    "comp.graphics",
    "talk.politics.misc"
]

dataset = fetch_20newsgroups(
    subset="train",
    categories=categories,
    remove=("headers", "footers", "quotes")
)

print("Número total de documentos:", len(dataset.data))
print("Categorías:", dataset.target_names)

Número total de documentos: 2239
Categorías: ['comp.graphics', 'rec.sport.baseball', 'sci.space', 'talk.politics.misc']


### Convertir corpus a DataFrame

In [5]:
df_corpus = pd.DataFrame({
    "doc_id": range(len(dataset.data)),
    "text": dataset.data,
    "category_id": dataset.target
})

df_corpus["category"] = df_corpus["category_id"].apply(
    lambda x: dataset.target_names[x]
)

df_corpus.head()

,doc_id,text,category_id,category
0,0,I thought that was Sandy Koufax.,1,rec.sport.baseball
1,1,"\nAnd the religious right worships engines, sm...",3,talk.politics.misc
2,2,\nHow can a witness tell that someone in a bur...,3,talk.politics.misc
3,3,"\n\n\n\nYes, long before Star Trek. Before Ei...",2,sci.space
4,4,\n\nIt depends. If you can get your old veter...,1,rec.sport.baseball


### Limpiar textos vacíos y limitar documentos

In [6]:
df_corpus["text"] = df_corpus["text"].fillna("").str.strip()

df_corpus = df_corpus[df_corpus["text"] != ""].reset_index(drop=True)

MAX_DOCS = 10

df_corpus = df_corpus.head(MAX_DOCS).copy()

print("Documentos usados:", len(df_corpus))
df_corpus.head()

Documentos usados: 10


,doc_id,text,category_id,category
0,0,I thought that was Sandy Koufax.,1,rec.sport.baseball
1,1,"And the religious right worships engines, smok...",3,talk.politics.misc
2,2,How can a witness tell that someone in a burni...,3,talk.politics.misc
3,3,"Yes, long before Star Trek. Before Einstein, ...",2,sci.space
4,4,It depends. If you can get your old veterans ...,1,rec.sport.baseball


### 2.2 Transformo a embeddings

### Funcion para obtener embedings

In [7]:
def get_embedding(text, task_type="RETRIEVAL_DOCUMENT"):
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text,
        config=types.EmbedContentConfig(
            task_type=task_type
        )
    )
    
    return np.array(response.embeddings[0].values)

### Generar embedings

In [8]:
from tqdm.notebook import tqdm

document_embeddings = []

for text in tqdm(df_corpus["text"], total=len(df_corpus)):
    emb = get_embedding(text, task_type="RETRIEVAL_DOCUMENT")
    document_embeddings.append(emb)

document_embeddings = np.vstack(document_embeddings)

print("Shape de embeddings:", document_embeddings.shape)

  0%|          | 0/10 [00:00<?, ?it/s]

Shape de embeddings: (10, 3072)


### 2.3 Creo una query y hago la búsqueda

### Crear query

In [9]:
query = "What is NASA planning to do with Earth and space science data?"

query_embedding = get_embedding(query, task_type="RETRIEVAL_QUERY")

print("Embedding de la query:", query_embedding.shape)

Embedding de la query: (3072,)


### Calcular similitud coseno

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(
    query_embedding.reshape(1, -1),
    document_embeddings
)[0]

df_corpus["similarity"] = similarities

df_results = df_corpus.sort_values(
    by="similarity",
    ascending=False
).head(5)

df_results[["doc_id", "category", "similarity", "text"]]

,doc_id,category,similarity,text
8,9,sci.space,0.742745,==============================================...
3,3,sci.space,0.573098,"Yes, long before Star Trek. Before Einstein, ..."
7,8,sci.space,0.565006,"As I heard the story, before Albert came up th..."
1,1,talk.politics.misc,0.555420,"And the religious right worships engines, smok..."
4,4,rec.sport.baseball,0.541733,It depends. If you can get your old veterans ...


In [11]:
for idx, row in df_results.reset_index(drop=True).iterrows():
    print("=" * 80)
    print(f"Documento {idx + 1}")
    print("Doc ID:", row["doc_id"])
    print("Categoría:", row["category"])
    print("Similitud:", row["similarity"])
    print()
    print(row["text"][:1000])

Documento 1
Doc ID: 9
Categoría: sci.space
Similitud: 0.7427453976119911

I am posting this for someone else.  Please respond to the 
address listed below.  Please also excuse the duplication as this 
message has been crossposted.  Thanks!
 
 
      REQUEST FOR IDEAS FOR APPLICATIONS OF REMOTE SENSING DATABASES 
                             VIA THE INTERNET
 
NASA is planning to expand the domain of users of its Earth and space science
data.  This effort will:
 
  o   Use the evolving infrastructure of the U.S. Global Change Research 
      Program including the Mission To Planet Earth (MTPE) and the Earth 
      Observing System Data and Information System (EOSDIS) Programs.
 
  o   Use the Internet, particularly the High Performance Computing and 
      Communications Program's NREN (National Research and Education 
      Network), as a means of providing access to and distribution of 
      science data and images and value a
Documento 2
Doc ID: 3
Categoría: sci.space
Similitud: 0.5

## Usar Gemini

In [12]:
resultado_ir = "\n\n".join(
    [
        f"Documento {idx + 1}:\n{row['text'][:1500]}"
        for idx, row in df_results.reset_index(drop=True).iterrows()
    ]
)

print(resultado_ir[:3000])

Documento 1:
I am posting this for someone else.  Please respond to the 
address listed below.  Please also excuse the duplication as this 
message has been crossposted.  Thanks!
 
 
      REQUEST FOR IDEAS FOR APPLICATIONS OF REMOTE SENSING DATABASES 
                             VIA THE INTERNET
 
NASA is planning to expand the domain of users of its Earth and space science
data.  This effort will:
 
  o   Use the evolving infrastructure of the U.S. Global Change Research 
      Program including the Mission To Planet Earth (MTPE) and the Earth 
      Observing System Data and Information System (EOSDIS) Programs.
 
  o   Use the Internet, particularly the High Performance Computing and 
      Communications Program's NREN (National Research and Education 
      Network), as a means of providing access to and distribution of 
      science data and images and value added products.
 
  o   Provide broad access to and utilization of remotely sensed images in 
      cooperation with oth

In [13]:
prompt_final = f"""
Eres un asistente estricto de Recuperación de Información.

Mi usuario quiere saber:
"{query}"

Mi Sistema de Recuperación de Información encontró esto:
"{resultado_ir}"

Dame la respuesta en base únicamente al contexto dado por el Sistema de Recuperación de Información.

REGLAS CRÍTICAS:
1. Responde solo usando la información recuperada por el Sistema de Recuperación de Información.
2. Si la respuesta exacta no está en el resultado recuperado, responde textualmente:
"Lo siento, no tengo esa información en mi base de conocimientos."
3. No inventes, no asumas y no uses conocimiento externo.
4. Sé directo, breve y objetivo.

RESPUESTA:
"""

In [14]:
try:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt_final,
        config=types.GenerateContentConfig(
            temperature=0.0
        )
    )

    print("Respuesta de Gemini:")
    print(response.text)

except Exception as e:
    print(f"Ocurrió un error: {e}")

Respuesta de Gemini:
NASA está planeando expandir el dominio de usuarios de sus datos de ciencia de la Tierra y el espacio. Este esfuerzo:
*   Usará la infraestructura en evolución del Programa de Investigación del Cambio Global de EE. UU., incluyendo los Programas Mission To Planet Earth (MTPE) y Earth Observing System Data and Information System (EOSDIS).
*   Usará Internet, particularmente la NREN (National Research and Education Network) del Programa de Computación y Comunicaciones de Alto Rendimiento, como un medio para proporcionar acceso y distribución de datos e imágenes científicas y productos de valor añadido.
*   Proporcionará amplio acceso y utilización de imágenes de teledetección en cooperación con otras agencias (especialmente NOAA, EPA, DOE, DEd, DOI/USGS y USDA).
*   Apoyará a los usuarios de imágenes y datos de teledetección y a las comunidades de desarrollo.
Las comunidades de usuarios y desarrollo a incluir (pero no limitadas a) como parte de este esfuerzo son educa